# Quantum Mechanics Basics

## Imports

In [1]:
# Core Qiskit imports
from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# IBM Runtime specific imports
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# DotEnv and OS
from dotenv import load_dotenv
import os

In [2]:
import sys

import qiskit
import qiskit_aer
import qiskit_ibm_runtime
import pylatexenc

print("Python:", sys.version.split()[0])
print("qiskit:", qiskit.__version__)
print("qiskit-aer:", qiskit_aer.__version__)
print("qiskit-ibm-runtime:", qiskit_ibm_runtime.__version__)
print("pylatexenc:", pylatexenc.__version__)

Python: 3.11.18
qiskit: 2.6.2
qiskit-aer: 0.18.2
qiskit-ibm-runtime: 0.52.1
pylatexenc: 2.11


## Helper Functions

In [4]:

import os
from dotenv import load_dotenv
from qiskit_ibm_runtime import QiskitRuntimeService


def join_least_busy_backend(min_num_qubits=127):
    """
    Connects to IBM Quantum and returns the least busy
    operational, non-simulator backend.

    Args:
        min_num_qubits (int): Minimum number of physical qubits.

    Returns:
        Backend: Selected IBM Quantum backend.
    """
    load_dotenv()

    IBM_TOKEN = os.getenv("IBM_QUANTUM_TOKEN")

    if not IBM_TOKEN:
        raise ValueError("IBM_QUANTUM_TOKEN not found in .env")

    # Initialize service using the environment token
    service = QiskitRuntimeService(
        channel="ibm_quantum_platform",
        token=IBM_TOKEN,
    )

    # Find least busy physical backend
    backend = service.least_busy(
        operational=True,
        simulator=False,
        min_num_qubits=min_num_qubits,
    )

    print(f"Connected to backend: {backend.name}")
    print(f"Number of qubits: {backend.num_qubits}")

    return backend

In [5]:
def run_circuit_and_get_counts(circuit, backend, shots=1000):
    """
    Runs a quantum circuit on a specified backend and returns the measurement counts.

    Args:
        circuit (QuantumCircuit): The quantum circuit to run.
        backend: The Qiskit backend (real device or simulator).
        shots (int): The number of shots to run the circuit.

    Returns:
        dict: A dictionary of measurement counts.
    """
    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    isa_circuit = pm.run(circuit)

    sampler = Sampler(mode=backend)

    job = sampler.run([isa_circuit], shots=shots)
    result = job.result()

    return result[0].data.meas.get_counts()

## Tutorial